In [ ]:
import pandas as pd
import numpy as np

CURRENT_YEAR = 2026

# ============================================================
# LOAD FILES
# ============================================================

df = pd.read_csv("../data/drivearabia_car_depreciation_valuation/original/mazda.csv")
dep_df = pd.read_csv("../data/drivearabia_car_depreciation_valuation/annual_dep_rate.csv")

# ============================================================
# CLEAN DEPRECIATION DATA
# ============================================================

dep_df["make"] = dep_df["make"].astype(str).str.strip().str.upper()
dep_df["model"] = dep_df["model"].astype(str).str.strip().str.upper()

# ============================================================
# MAZDA MODEL MAPPING
# ============================================================

MODEL_MAPPING = {
    "mazda-2": "MAZDA2",
    "mazda-3": "MAZDA3",
    "mazda-6": "MAZDA6",
    "mazda-cx-3": "CX-3",
    "mazda-cx-30": "CX-30",
    "mazda-cx-5": "CX-5",
    "mazda-cx-60": "CX-60",
    "mazda-cx-9": "CX-9",
    "mazda-cx-90": "CX-90",
    "mazda-mx-5": "MX-5",
    "mazda-bt-50": "BT-50",
}

df["make"] = "MAZDA"

df["model_name"] = (
    df["model_slug"]
    .map(MODEL_MAPPING)
    .astype(str)
    .str.upper()
)

# ============================================================
# BUILD LOOKUP
# ============================================================

dep_lookup = (
    dep_df.groupby(["make", "model"])["annual_dep_rate"]
    .mean()
    .reset_index()
)

# ============================================================
# MERGE DEPRECIATION RATE
# ============================================================

df = df.merge(
    dep_lookup,
    left_on=["make", "model_name"],
    right_on=["make", "model"],
    how="left"
)

# ============================================================
# CAR AGE
# ============================================================

df["car_age"] = CURRENT_YEAR - df["year"]
df["car_age"] = df["car_age"].clip(lower=0)

# ============================================================
# DEPRECIATED VALUE
# ============================================================

def calc_depreciated_value(row):

    rate = row["annual_dep_rate"]

    if pd.isna(rate):
        return np.nan

    age = row["car_age"]

    if age == 0:
        return round(row["price_avg_aed"], 0)

    return round(
        row["price_avg_aed"] * ((1 - rate) ** age),
        0
    )

df["depreciated_value"] = df.apply(
    calc_depreciated_value,
    axis=1
)

# ============================================================
# VALIDATION
# ============================================================

print("Total Rows:", len(df))
print("Matched Rates:", df["annual_dep_rate"].notna().sum())
print("Missing Rates:", df["annual_dep_rate"].isna().sum())

print("\nUnmatched Models:")
print(
    df[df["annual_dep_rate"].isna()]
    ["model_slug"]
    .drop_duplicates()
    .tolist()
)

# ============================================================
# SAVE
# ============================================================

df.to_csv(
    "../data/drivearabia_car_depreciation_valuation/depreciated/mazda_dep.csv",
    index=False
)

print("\nSaved: ../data/drivearabia_car_depreciation_valuation/depreciated/mazda_dep.csv")

Total Rows: 92
Matched Rates: 55
Missing Rates: 37

Unmatched Models:
['mazda-2', 'mazda-3', 'mazda-3-sedan', 'mazda-6']

Saved: data/mazda_dep.csv
